# Day 1 Smoke Test (Colab Pro+ A100)

Validates the Day 1 pipeline before the overnight corpus run. Four cells:
1. Mount Drive, clone repo, install deps, symlink `data/` to Drive.
2. Load Qwen2.5-7B and run a forward pass; confirm `n_layers` and `hidden_dim`.
3. One Level 0 (clean) trajectory end-to-end with activation capture.
4. One Level 1 + one Level 2 trajectory; eyeball the perturbations.

If any of (2)–(4) fails, do **not** start the overnight run. See `project_plan.md` §12 for fallbacks.

## 1. Mount Drive, clone, install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/agent_faithfulness'
REPO_DIR  = '/content/agent_faithfulness'
GITHUB_URL = 'https://github.com/<YOUR_USERNAME>/agent_faithfulness.git'  # <-- EDIT ME

import os, subprocess
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data/activations', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', GITHUB_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull'])

%cd $REPO_DIR
!pip install -q -r requirements.txt

## 2. Model load + forward pass

In [ ]:
import sys, os
sys.path.insert(0, REPO_DIR)

import torch
from src.agent import load_model

model, tok = load_model('Qwen/Qwen2.5-7B-Instruct', dtype='bfloat16')

n_layers = len(model.model.layers)
hidden_dim = model.config.hidden_size
device = next(model.parameters()).device
dtype  = next(model.parameters()).dtype
print(f'device={device} dtype={dtype} n_layers={n_layers} hidden_dim={hidden_dim}')

# One forward pass to verify generation works.
ids = tok('Hello', return_tensors='pt').input_ids.to(device)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=True))

## 3. One Level 0 trajectory end-to-end

In [ ]:
import random, json, torch
from pathlib import Path
from src.catalog import generate_catalog
from src.queries import sample_query
from src.agent import run_trajectory
from src.labels import compute_all_labels

ACTS_DIR = Path(f'{DRIVE_ROOT}/data/activations_smoke')
ACTS_DIR.mkdir(parents=True, exist_ok=True)

catalog = generate_catalog(n=200, seed=42)
rng = random.Random(0)
query = sample_query(catalog, n_constraints=3, rng=rng, query_id='query_smoke_0')
print('Query:', query['natural_language'])
print('Constraints:', query['constraints'])

traj = run_trajectory(
    model, tok,
    catalog=catalog, query=query,
    perturbation_level=0, rng=rng,
    trajectory_id='traj_smoke_l0',
    max_steps=5, capture_activations=True,
    activations_dir=ACTS_DIR,
)
compute_all_labels(traj)

print('\nfinal_action:', traj['final_action'])
print('final_item:', traj['final_item'])
print('labels:', traj['labels'])
print('n_steps:', len(traj['steps']))
for s in traj['steps']:
    print(f"  step {s['step_idx']} thought-len={len(s['thought'])} "
          f"tool_call={'yes' if s['tool_call'] else 'no'} acts={s['activations_path']}")

# Verify activation shape on at least one captured step.
act_files = list(ACTS_DIR.glob('traj_smoke_l0_*.pt'))
assert act_files, 'No activation files written.'
t = torch.load(act_files[0])
print(f'\nactivation shape: {tuple(t.shape)}, dtype={t.dtype}')
assert t.shape == (n_layers, hidden_dim), f'Bad shape: {t.shape}'
assert traj['final_action'] is not None and traj['final_action'].startswith('PURCHASE'), \
    'Level 0 must end with a PURCHASE.'
print('\nLevel 0 smoke: PASS')

## 4. Level 1 + Level 2 trajectories

In [ ]:
for level in (1, 2):
    rng_local = random.Random(level)
    q = sample_query(catalog, n_constraints=3, rng=rng_local, query_id=f'query_smoke_l{level}')
    traj = run_trajectory(
        model, tok,
        catalog=catalog, query=q,
        perturbation_level=level, rng=rng_local,
        trajectory_id=f'traj_smoke_l{level}',
        max_steps=5, capture_activations=True,
        activations_dir=ACTS_DIR,
    )
    compute_all_labels(traj)
    print(f'\n=== Level {level} ===')
    print('Query:', q['natural_language'])
    print('Constraints:', q['constraints'])
    print('perturbed_attribute:', traj['perturbed_attribute'])
    print('original → perturbed:', traj['original_value'], '→', traj['perturbed_value'])
    print('final_action:', traj['final_action'])
    print('labels:', traj['labels'])
    # Eyeball: confirm tool_result[0] differs from tool_result_clean[0] at the perturbation step.
    for s in traj['steps']:
        if s['tool_result'] and s['tool_result_clean']:
            tr0 = s['tool_result'][0]
            trc0 = s['tool_result_clean'][0]
            if tr0 != trc0:
                print(f"  step {s['step_idx']} top differs:")
                print('    perturbed:', tr0)
                print('    clean    :', trc0)
                break
print('\nLevel 1/2 smoke: review the prints above before kicking off overnight run.')